# Water Potability Prediction Using Machine Learning

## 1. Introduction

Access to safe drinking water is essential for human health and environmental sustainability.<br> 
Water quality is tipically evaluated using several physicochemical parameters such as pH, Hardness, Solids, and Turbidity.<br>
Machine learning techniques can be used to analyse these parameters and help predict whether a water sample is potable or not.

## 2. Objective

The objective of this project is to evaluate the performance of different machine learning models to predict whether water is safe for human consumption based on physicochemical water quality parameters.

## 3. Dataset Description

The dataset used in this project comes from the **Water Potability Dataset** available on Kaggle.<br>
Each observation represents a water sample described by several physicochemical parameters related to water quality.

The dataset includes the following variables:
- pH
- Hardness
- Solids
- Chloramines
- Sulfate
- Conductivity
- Organic carbon
- Trihalomethanes
- Turbidity

Target variable:
- **Potability**
   - **1** = Potable (safe for drinking)
   - **0** = Not potable

## 4. Import Libraries 

This section imports the Python libraries required for data analysis, visualisation and machine learning.<br> 
These libraries provide the functions and tools used throughout the notebook to explore the dataset, process the data and build predictive models.

In [1]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualisation
import matplotlib.pyplot as plt 
import seaborn as sns

# Statistical Analysis (Variance Inflation Factor - VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Feature Importance (Mutual Information)
from sklearn.feature_selection import mutual_info_classif

# Train-Test Split
from sklearn.model_selection import train_test_split

# Feature Scaling
from sklearn.preprocessing import StandardScaler

# Logistic Regression (LR)
from sklearn.linear_model import LogisticRegression

# K-Nearest Neighbors (KNN)
from sklearn.neighbors import KNeighborsClassifier

# Support Vector Machine (SVM)
from sklearn.svm import SVC

# Random Forest (RF)
from sklearn.ensemble import RandomForestClassifier

# Gradient Boosting (GB)
from sklearn.ensemble import GradientBoostingClassifier

# Model Evaluation Metrics
from sklearn.metrics import accuracy_score         # accuracy
from sklearn.metrics import classification_report  # classification report
from sklearn.metrics import confusion_matrix       # confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay # confusion matrix analysis

# Receiver Operating Characteristic (ROC Curve) and Area Under de Curve (AUC Score)
from sklearn.metrics import roc_curve, roc_auc_score

# Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV   # Best Hyperparameters

import warnings
warnings.filterwarnings("ignore")

## 5. Exploratory Data Analysis (EDA)

### 5.1 Dataset Overview

In this step the dataset is loaded and the first rows are displayed to understand the structure of the data.

In [3]:
water = pd.read_csv('water_potability.csv')
water

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0
...,...,...,...,...,...,...,...,...,...,...
3271,4.668102,193.681735,47580.991603,7.166639,359.948574,526.424171,13.894419,66.687695,4.435821,1
3272,7.808856,193.553212,17329.802160,8.061362,NaN,392.449580,19.903225,NaN,2.798243,1
3273,9.419510,175.762646,33155.578218,7.350233,NaN,432.044783,11.039070,69.845400,3.298875,1
3274,5.126763,230.603758,11983.869376,6.303357,NaN,402.883113,11.168946,77.488213,4.708658,1


The dataset contains **3,276 water samples** described by physicochemical parameters related to water potability.<br>
Each row represents a different water sample, while each column represents a specific water quality parameter. The column **Potability** is a **binary variable** that indicates whether the water is safe to drink (**1** = potable, **0** = no potable).<br>
Some variables contain missing values, which will be analysed and handled later during the data preprocessing stage.<br>
This dataset does not include spatial or temporal information, meaning that each sample is independent and does not represent a specific geographic location or time period.

### 5.2 Water Quality Parameters

The recommended ranges mentioned below are generally based on international drinking water guidelines such as the World Health Organization (WHO) Guidelines for Drinking-water Quality and the EU Drinking Water Directive. Values outside these ranges may indicate potential water quality problems and possible risks for human consumption.

**pH**<br>
pH measures the acidity or alkalinity of water and reflects the concentration of hydrogen ions present in the sample.<br>
According to international drinking water guidelines, the recommended range is typically **6.5 – 8.5**.<br>
Water with a low pH (acidic) may become corrosive and can dissolve metals from pipes and plumbing systems, including trace metals such as lead, copper and iron, which may contaminate drinking water and pose potential health risks.<br> 
Water with a high pH (alkaline) may produce an unpleasant taste and promote the formation of mineral deposits (scale) in pipes and household appliances.<br>
Extreme pH levels may also irritate the eyes, skin and mucous membranes, which may reduce the suitability of water for human consumption.

**Hardness**<br>
Hardness refers to the concentration of dissolved calcium (Ca²⁺) and magnesium (Mg²⁺) ions in water.<br>
Typical drinking water values commonly range between **50 and 150 mg/L**.<br>
Very hard water can cause scaling in pipes, boilers and household appliances.<br> 
Conversely, very soft water may increase the corrosion of plumbing systems.<br>
Although hardness generally does not pose a direct health risk, very high levels may affect taste and water acceptability for consumption.

**Solids**<br>
Total dissolved solids (TDS) represent the combined amount of dissolved substances in water, including minerals, salts and trace metals.<br>
Drinking water guidelines generally recommend concentrations **below 500 mg/L**.<br>
High levels of dissolved solids may affect the taste of water and may indicate elevated mineral content or potential contamination from natural or anthropogenic sources.

**Chloramines**<br>
Chloramines are disinfectants formed when chlorine reacts with ammonia during water treatment. They are commonly used to control microbial growth in drinking water systems.<br>
Typical concentrations are generally **below 4 mg/L**.<br>
Elevated chloramine levels may cause unpleasant taste and odour and may irritate the eyes or skin. Excessive concentrations can also reduce the acceptability of water for drinking.

**Sulfate**<br>
Sulfate refers to the concentration of sulfate ions (SO₄²⁻) present in water.<br>
Drinking water guidelines commonly recommend concentrations **below 250 mg/L**.<br>
High sulfate levels may produce a bitter taste and can have a laxative effect, particularly for individuals who are not accustomed to such concentrations.

**Conductivity**<br>
Conductivity measures the ability of water to conduct electricity, which depends on the concentration of dissolved ions such as salts and minerals.<br>
Typical drinking water values are usually **below 500 µS/cm**.<br>
High conductivity may indicate elevated concentrations of dissolved salts or possible contamination from natural or anthropogenic sources.

**Organic Carbon**<br>
Organic carbon represents the amount of organic matter present in water from natural sources or human activities.<br>
Recommended concentrations are generally **below 2 mg/L**.<br>
High organic carbon levels may react with disinfectants during water treatment and form chemical by-products such as Trihalomethanes. Long-term exposure to these compounds may affect the liver and kidneys and may increase potential health risks.

**Trihalomethanes**<br>
Trihalomethanes (THMs) are chemical by-products formed when chlorine used for water disinfection reacts with natural organic matter.<br>
Recommended concentrations are generally **below 80 µg/L**.<br>
High concentrations of trihalomethanes have been associated with potential long-term health risks such as liver problems, kidney effects and an increased risk of certain cancers when consumed over long periods.

**Turbidity**<br>
Turbidity measures the cloudiness of water caused by suspended particles such as silt, microorganisms or organic matter.<br>
Drinking water should ideally have turbidity levels **below 1 NTU**.<br>
High turbidity may reduce the effectiveness of disinfection processes and may indicate possible microbial contamination, which can lead to gastrointestinal illnesses such as diarrhoea or stomach infections.

**Potability**<br>
Potability indicates whether the water is considered safe for human consumption.<br>
In this dataset:<br>
    - **1** = potable water  
    - **0** = non-potable water<br>
Water is generally considered potable when key water quality parameters fall within recommended guideline values. If one or more parameters exceed these limits, the water may not meet drinking water standards and may be considered unsafe for consumption.

### 5.3 Data Structure and Data Types

This section examines the general structure of the dataset and the data types of each variable in order to understand how the information is organised before further analysis.

#### 5.3.1 General Structure of the Dataset

In [5]:
water.shape

(3276, 10)

In [6]:
water.columns

Index(['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity',
       'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability'],
      dtype='object')

In [7]:
water.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3276 entries, 0 to 3275
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ph               2785 non-null   float64
 1   Hardness         3276 non-null   float64
 2   Solids           3276 non-null   float64
 3   Chloramines      3276 non-null   float64
 4   Sulfate          2495 non-null   float64
 5   Conductivity     3276 non-null   float64
 6   Organic_carbon   3276 non-null   float64
 7   Trihalomethanes  3114 non-null   float64
 8   Turbidity        3276 non-null   float64
 9   Potability       3276 non-null   int64  
dtypes: float64(9), int64(1)
memory usage: 256.1 KB


The function `shape` shows the dimensions of the dataset, indicating the number of rows and columns. In this case, the dataset contains **3,276 rows and 10 columns**.<br>
The function `columns` displays the names of all variables included in the dataset.<br>
Finally, the function `info()` provides a summary of the dataset, including the number of observations, column names, and the number of non-null values for each variable.

#### 5.3.2 Data Types of Each Column

In [8]:
water.dtypes  # astype() is used to change the data type of a single column

ph                 float64
Hardness           float64
Solids             float64
Chloramines        float64
Sulfate            float64
Conductivity       float64
Organic_carbon     float64
Trihalomethanes    float64
Turbidity          float64
Potability           int64
dtype: object

The output shows the data type associated with each column in the dataset.<br>
**All physicochemical parameters** are numerical continuous variables stored as **floating-point values**, while the target variable **Potability** is stored as an **integer**.<br>
Since the variables are already stored using appropriate data types, no conversion is required. However, if a variable were incorrectly formatted, the function `astype()` could be used to convert it to the appropriate data type.ype.

### 5.4 Descriptive Statistics 

### 3.1) Descriptive Statistics of Numerical Variables

In [ ]:
water.describe()

#### The descriptive statistics show that most variables present reasonable ranges according to their physical and chemical meaning.
#### Based on the minimum and maximum values, no negative values were detected in the dataset, which indicates consistency since all measured water quality parameters must be non-negative.
#### Additionally, the pH values range between 0 and 14, which corresponds to the theoretical and physically acceptable scale of pH measurements. This suggests that there are no obvious data entry errors in this variable.
#### Some features such as solids and conductivity show small differences between mean and median values, which indicates a slight skewness in their distributions. However, the differences are not substantial and may reflect natural variability in water quality measurements rather than data anomalies.
#### Overall, the initial descriptive analysis suggests that the dataset does not contain major inconsistencies that would require immediate data correction or removal.

### 3.2) Distribution Analysis

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(water['ph'], bins=14, range=(0,14), edgecolor='black')
plt.title('pH Histogram')
plt.xlabel('ph')
plt.ylabel('Frecuency')
plt.show()

In [ ]:
water.hist(figsize=(14,12), bins=10, edgecolor='black')

In [ ]:
for col in water.columns:
    plt.figure(figsize=(5,3))
    plt.hist(water[col], bins=14, edgecolor='black')
    plt.title(f'{col} Histogram')
    plt.xlabel(col)
    plt.ylabel('Frecuency')
    plt.show()

In [ ]:
for col in water.columns:
    water[col].skew()
    print ([col], water[col].skew())

In [ ]:
for col in water.columns:
    print (col, 'Skewness:', [col], water[col].skew())

#### The skewness analysis indicates that most variables present values close to zero, suggesting approximately symmetric distributions. 
#### Some variables, such as solids and conductivity, show slight positive skewness, indicating the presence of higher values in the upper range of the distribution. However, the magnitude of skewness across variables remains relatively small (mostly between -0.5 and 0.5), suggesting that the distributions are reasonably balanced and do not exhibit strong asymmetry.
#### The potability variable was excluded from interpretation since it represents a binary target variable rather than a continuous measurement.

In [ ]:
sns.boxplot(x=water['ph'])

In [ ]:
water.drop(columns='Potability').boxplot(figsize=(12,6),rot=45)

In [ ]:
for col in water.columns:
    water.boxplot(column=col, figsize=(4,3))
    plt.title(col)
    plt.show()

#### Boxplot analysis shows the presence of outliers across most variables. This is expected in environmental datasets where water quality parameters can vary significantly depending on geographic and contamination conditions. Therefore, these values are not immediately considered erroneous but will be further evaluated during preprocessing.

### 3.3) Identifying Potential Outliers

#### 3.3.1) IQR (Interquartile Range)
#### It uses quartiles and works best with skewed data

In [ ]:
# Show all columns from the rows where pH is an outlier
Q1 = water['ph'].quantile(0.25)
Q3 = water['ph'].quantile(0.75)
IQR = Q3-Q1
lower = Q1-1.5*IQR 
upper = Q3+1.5*IQR 

outliers = water[(water['ph']<lower) | (water['ph']>upper)]
print(outliers)

In [ ]:
Q1 = water['ph'].quantile(0.25)
Q3 = water['ph'].quantile(0.75)
IQR = Q3-Q1
lower = Q1-1.5*IQR 
upper = Q3+1.5*IQR 

outliers = water.loc[(water['ph']<lower) | (water['ph']>upper),'ph']
print(outliers)

In [ ]:
for col in water.drop(columns='Potability').columns:
    data = water[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3-Q1
    lower = Q1-1.5*IQR 
    upper = Q3+1.5*IQR 
    outliers = data[(data<lower) | (data>upper)]
    print(col, 'outliers:', len(outliers))

#### 3.3.2) Z-score 
#### It uses mean and standard deviation, and works best with normal data

In [ ]:
# z = (value-mean)/std
# |z|>3 = outlier
from scipy.stats import zscore
z_scores = np.abs(zscore(water.drop(columns='Potability').dropna()))
outliers = (z_scores>3).sum(axis=0)
print(outliers)

#### 3.3.3) Percentile-based methods (1%. 99%)
#### It uses a fixed percentage and works best with practical control

In [ ]:
for col in water.drop(columns='Potability').columns:
    data = water[col].dropna()
    lower = data.quantile(0.01)
    upper = data.quantile(0.99)
    outliers = data[(data<lower) | (data>upper)]
    print(col, 'percentile outliers:', len(outliers))

#### Outliers were identified using three different approaches: IQR, Z-score, and percentile-based methods. Each technique produced different counts of extreme values, which is expected due to their distinct statistical assumptions.

#### The IQR method (+) identified the highest number of outliers, likely because it is robust to skewed distributions, which are common in environmental datasets. The Z-score method (-) detected fewer outliers, as it assumes approximately normal distributions and relies on mean-based deviation. Percentile analysis (+) identified extreme values at the tails of the distribution without assuming any specific data shape.

#### Given the environmental nature of the dataset, these outliers are not automatically considered erroneous measurements. Instead, they may represent real variability in water quality conditions. Therefore, no immediate removal of outliers was performed at this stage. Further evaluation will be conducted during data preprocessing.

## 4) Data Quality and Cleaning

### 4.1) Dataset Structure Verification

In [ ]:
# Verify dataset shape before cleaning
print("Dataset shape before cleaning:", water.shape)

# Verify data types
print("\nData types:")
print(water.dtypes)

# Check for non-numeric columns
non_numeric = water.select_dtypes(exclude=["int64", "float64"])
print("\nNon-numeric columns:")
print(non_numeric.columns)

#### The dataset structure was verified prior to cleaning procedures. The dataset contains 3276 observations and 10 variables. All predictor variables are numerical (float64), and the target variable (Potability) is encoded as an integer (int64). No non-numeric columns were detected, indicating that no type conversion or categorical encoding is required at this stage


### 4.2) Duplicate Records Analysis

In [ ]:
# Count duplicated rows
duplicate_count = water.duplicated().sum()

print("Number of duplicated rows:", duplicate_count)

#### A dupicate record analysis was conducted to ensure data integrity. No duplicated rows were identified in the dataset, indicating that each observation represents a unique water sample.

### 4.3) Missing Values Analysis

#### 4.3.1) Detection of Missing Values

In [ ]:
missing_total = water.isnull().sum().sum()

print("Total missing values in dataset:", missing_total)

#### A total of 1,434 missing values were detected across the dataset. This indicates the presence of incomplete observations that must be addressed before model training, as machine learning algorithms generally require complete numerical input.

#### 4.3.2) Missing values per column

In [ ]:
missing_per_column = water.isnull().sum()
print(missing_per_column)

#### Missing values are concentrated in three variables: pH (491), Sulfates (781), and Trihalomethanes (162), while the remaining features contain no missing observations. This suggests that missingness is not randomly distributed across all variables, but instead affects specific chemical parameters.
#### The distribution of missing values will be further explored to determine the most appropriate handling strategy.y.






#### 4.3.3) Missing values per row

In [ ]:
missing_per_row = water.isnull().sum(axis=1)

print("Maximum missing values in a single row:", missing_per_row.max())
print("Rows with at least one missing value:", (missing_per_row > 0).sum())

#### The maximum number of missing values observed in a single row is 3 and 1265 rows contain at least one missing value. This indicates that missing data affects a substantial portion of observations; however, no row presents excessive missingness.

In [ ]:
missing_per_row = water.isnull().sum(axis=1)
missing_per_row.value_counts().sort_index()

#### The distribution of missing values per row shows that 2011 observations are complete, 1105 rows contain one missing value, 151 rows contain two missing values and only 9 rows contain three missing values.
#### This pattern indicates that missing values are dispersed across observations rather than concentrated within a small subset of rows. Therefore, removing entire rows would result in unnecessary data loss. Imputation appears to be a more appropriate strateg

#### 4.3.4) Percentage of missing values 

In [ ]:
missing_percentage = water.isnull().sum() / water.shape[0]*100
missing_percentage.sort_values(ascending=False)

#### Sulfate has 23.84% missing values, pH has 14.98% and Trihalomethanes has 4.94%. Percentages indicate the proportion of rows affected relative to the total dataset size (3276 rows).
#### The percentage of missing values per column shows that some variables have a significant porportion of missing data, while the other columns are complete. This will be considered when deciding how to handle missing data. ons.

#### 4.3.5) Visualization of missing data

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(water.isnull(), cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

#### Visualization reveals patterns in missing data, helping to decide if amputation or row removal is more appropiate.
#### The missing value matrix shows that missing data is mainly present in pH, Sulfate, and Trihalomethanes. The missing values appear to be randomly distributed across observations rather than concentrated in specific rows, so imputation is suitable.
#### Additionally, the missing values correlation heatmap indicates weak relationships between missing patterns across variables, suggesting that missingness is likely independent between variables.

### 4.4) Missing Value Handling Strategy

#### Several approaches can be used to handle missing data, including row deletion, column removal, and value imputation.
#### Analysis of missing values per row revealed that missing data is scattered rather than concentrated. The maximum number of missing values observed in a single row was 3 out of 10 variables, indicating that most observations still contain sufficient information for analysis. Therefore, row removal was not considered appropriate. Furthermore, removing rows containing missing values would significantly reduce the dataset size. In this dataset, 1434 rows contain missing values. Since these variables are chemically important for water quality analysis, removing them would reduce the available data for model training and potencially affect model performance. Therefore, removing rows was not considered appropriate.
#### Column removal is generally considered when a variable contains more than 40–50% missing values, as such high proportions may reduce data reliability. However, in this dataset, the highest percentage of missing data corresponds to Sulfates (23.84%), followed by pH (14.98%) and Trihalomethanes (4.94%). Since these percentages are below commonly accepted removal thresholds, all variables were retained. Additionally, these parameters are chemically important indicators of water quality, making their removal scientifically inappropriate.
#### Additionally, visualization of missing values using heatmaps suggests that missing data is randomly distributed across observations. This indicates that imputation is a suitable approach.
#### Distribution analysis performed during exploratory data analysis showed that most variables exhibit approximately symmetric distributions with mild skewness and the presence of outliers, which is common in environmental datasets. Because mean values are sensitive to extreme observations, median imputation was selected as the preferred method. Median imputation provides a more robust central tendency estimate and reduces the influence of extreme environmental measurements.
#### Therefore, missing values in pH, Sulfates, and Trihalomethanes were handled using median imputation to preserve dataset size while maintaining statistical reliability.

#### 4.4.1) Median imputation

In [ ]:
water_imputation = water.copy()
for col in water_imputation.columns:
    if water_imputation[col].isnull().sum() > 0:
        water_imputation[col] = water_imputation[col].fillna(water_imputation[col].median())

#### 4.4.2) Potst-imputation verification

In [ ]:
water_imputation.isnull().sum()

In [ ]:
water_imputation.head()

#### After applying imputation, all missing values were successfully replaced, as confirmed by the zero counts across all variables. The dataset is now complete and ready for further analysis and model development.

#### 4.4.3) Final dataset verification

In [ ]:
# Verify dataset shape after cleaning
print("Dataset shape bafter cleaning:", water_imputation.shape)

#### The dataset size remains unchanged after cleaning, with 3276 rows and 10 columns. This confirms that no observations were removed and only missing values were replaced using median imputation. The dataset is now complete and ready for further analysis.




## 5) Feature Relationships & Correlation Analysis

### 5.1) Correlation matrix

In [ ]:
water_imputation.corr()

#### Correlation
#### 0.0 - 0.3 weak
#### 0.3 - 0.7 moderate
#### 0.7 - 1.0 strong

In [ ]:
water_imputation.corr()['Potability']

#### The  correlation matrix was calculated to examine relationships between numerical variables. The results show that most correlations are weak, indicating limited linear relationships between features (A correlation less than |0.7| means that there are no strong linear relationships, the variables provide independent information and there is a low risk of multicollinearity). 
#### Additionally, correlations between individual features and the target variable (Potability) are generally low, suggesting that potability is influenced by multiple factors rather than a single dominant variable. 

### 5.2) Correlation heatmap

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(water_imputation.corr(), annot=True, cmap='coolwarm', fmt='.2f') # annot=True (show the numerical correlation values ​​within each cell) 
plt.title('Correlaton Heatmap')
plt.show()

#### The correlation heatmap provides a visual representation of relationships between variables. 
#### The correlation analysis shows very weak relationships among most variables, with the highest correlation observed between Sulfate and Solids (-0.15), which is still considered weak.
#### Similarly, correlations between individual features and the target variable (Potability) are close to zero, suggesting that water potability is influenced by multiple interacting factors rather than a single dominant parameter.
#### This suggests that each variable contributes unique information to the dataset, which can be beneficial for predictive modeling.

### 5.3) Feature vs. target analysis

In [ ]:
water_imputation.groupby('Potability').mean().T

In [ ]:
water_imputation.groupby('Potability').median().T

#### When the median and the mean are similar between potable and non-potable samples, it suggests that the variable alone has limited discriminative power in predicting water; therefore, that variable probably does not help predict potability.
#### The comparison of central tendency measures (mean and median) between potable and non-potable water samples shows minimal differences across most variables. 
#### Although Total Dissolved Solids (TDS) shows slightly higher values in potable water samples, the overlap between groups remains substantial. This suggests that water potability is likely influenced by the interaction of multiple variables rather than a single dominant factor.

In [ ]:
for col in water_imputation.drop(columns='Potability').columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='Potability', y=col, data=water_imputation)
    plt.title(f'{col} vs Potability')
    plt.show()

#### Boxplots are used to compare the distribution of each variable between potable and non-potable water samples. When the median values between groups are similar and the interquartile ranges overlap significantly, it suggests that the variable does not strongly differentiate between potable and non-potable water. Conversely, clear separation between medians and minimal overlap would indicate stronger predictive potential. In environmental datasets, the presence of outliers is common and typically reflects natural variability rather than data errors.
#### The boxplot analysis shows considerable overlap between potable and non-potable water samples across most variables. Median values are very similar between groups, suggesting weak individual relationships with water potability. Although some variables such as Total Dissolved Solids display slightly wider dispersion, no clear separation between classes is observed. This indicates that water potability "is likely influenced by a combination of variables" rather than a single dominant parameter..


### 5.4) Multicolinearity detection

#### 5.4.1) Correlation between features without target

In [ ]:
features = water_imputation.drop(columns='Potability')
corr_matrix = features.corr()
corr_matrix

#### corr > 0.8         high multicollinearity
#### corr (0.5 - 0.8)   moderate multicollinearity
#### corr < 0.5         low multicollinearity

In [ ]:
high_corr = np.where(abs(corr_matrix) > 0.8)

for i, j in zip(*high_corr):
    if i != j:
        print(corr_matrix.index[i], '-', corr_matrix.columns[j], corr_matrix.iloc[i, j])

#### No printed values ​​were observed because the results indicated that no pair of variables shows strong correlation above the commonly accepted threshold of 0.8, indicating low multicollinearity (Correlation measures the relationship between two variables)

#### 5.4.2) Heatmap for features

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(features.corr(), annot=True, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

#### 5.4.3) VIF Method (Variance Inflation Factor Method)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_data = pd.DataFrame()
vif_data["Feature"] = features.columns

vif_data["VIF"] = [variance_inflation_factor(features.values, i)
    for i in range(len(features.columns))]

vif_data

#### VIF measures how much "one variable can be explained by all the others together".
#### A high VIF means that this variable does not provide entirely new information; some of its information is already contained in other variables.
#### Multicollinearity works best with models like Random Forest, Decision Tree, XGBoost.
#### VIF < 5        very low multicollinearity
#### VIF (5 - 10)   moderate multicollinearity (potential concern)
#### VIF > 10       high multicollinearity (serious multicollinearity problem, redundant information)

#### Multicollinearity was evaluated using both correlation analysis and Variance Inflation Factor (VIF). Pairwise correlation analysis showed weak relationships between variables, indicating low direct multicollinearity. 
#### However, VIF results revealed higher values across several features, suggesting potential multicollinearity when considering combined interactions among variables. 
#### This pattern is expected in environmental datasets where chemical parameters naturally interact. Since tree-based machine learning models are robust to multicollinearity, all variables were retained for further analysis.

### 5.5) Scatter relationship analysis

#### 5.5.1) Scatterplot

In [ ]:
sns.scatterplot(data=water_imputation, x='Turbidity', y='Sulfate', hue='Potability')
plt.title('Turbidity vs Sulfate')
plt.xlabel('Turbidity')
plt.ylabel('Sulfate')
plt.show()

#### 5.5.2) Pairplot

In [ ]:
sns.pairplot(water_imputation, hue='Potability', diag_kind='kde')
plt.show()

#### Scatter plot and pairplot analysis were conducted to visually explore relationships between water quality parameters and potability. Most feature combinations showed widely dispersed point patterns without clear clustering or separation between potable and non-potable water samples. This suggests that individual variables may not strongly discriminate water potability and supports the hypothesis that potability is influenced by complex interactions among multiple parameters rather than single features.
#### Although visual separability between classes appears limited, this does not rule out the potential of machine learning models to capture non-linear and multi-feature interactions that may improve predictive performance.

## 6) Outliers Analysis

### 6.1) Outlier detection using statistical summary

#### 6.1.1) Statistical summary 

In [ ]:
water_imputation.describe()

#### A high standard deviation indicates a high level of variability in the data. In environmental datasets, this variability is expected due to natural fluctuations in water quality parameters.
#### The statistical summary indicates considerable variability across several water quality parameters. Variables such as Solids, Sulfate, and Conductivity show large ranges between minimum and maximum values, suggesting potential extreme observations. Additionally, high standard deviation values observed in some variables indicate substantial dispersion within the dataset. These patterns suggest the possible presence of outliers that require further evaluation.

#### 6.1.2) Explicit calculation of quartiles 

In [ ]:
Q1 = water_imputation.quantile(0.25)
Q3 = water_imputation.quantile(0.75)
IQR = Q3 - Q1

summary_outliers = pd.DataFrame({'Q1': Q1, 'Q3': Q3, 'IQR': IQR})
summary_outliers

#### The interquartile range (IQR) analysis reveals that Solids presents the largest variability, followed by Conductivity and Hardness. A larger IQR indicates that the central portion of the data is widely spread, suggesting higher variability in these parameters. 
#### This behavior is consistent with environmental water quality data, where certain chemical concentrations can fluctuate significantly depending on geographic and pollution-related factors.

#### 6.1.3) Detect extreme ranges

In [ ]:
range_values = water_imputation.max() - water_imputation.min()
range_values.sort_values(ascending=False)

#### Range analysis confirms that Solids exhibits the widest variation across the dataset, followed by Conductivity and Sulfate. Such large ranges may reflect natural environmental variability or localized water contamination events. Therefore, extreme values observed in these parameters should not be automatically classified as errors without further analysis.

### 6.2) Visual outlier detection

#### 6.2.1) Boxplot visualization

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=water_imputation.drop(columns='Solids'))
plt.title('Boxplot of Quality Variables')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=water_imputation[['ph','Chloramines','Organic_carbon','Turbidity']])
plt.title('Boxplot of Small Range Variables 1')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=water_imputation[['Hardness','Sulfate','Conductivity','Trihalomethanes']])
plt.title('Boxplot of Small Range Variables 2')
plt.show()

In [ ]:
for col in water_imputation.drop(columns='Potability'):
    water_imputation.boxplot(column=col, figsize=(4,3))
    plt.title(col)
    plt.show()

In [ ]:
# NUMBER OF OUTLIERS

outlier_summary = {}   
for col in water_imputation.drop(columns='Potability').columns:
    
    data = water_imputation[col]
    
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = data[(data < lower) | (data > upper)] 
    outlier_summary[col] = len(outliers)

pd.Series(outlier_summary).sort_values(ascending=False)

#### Boxplot visualizations were used to examine the distribution and spread of water quality parameters. The plots highlight the presence of extreme values across several variables. Based on the visualizations and the numerical analysis of outliers, Sulfate, pH, and Hardness exhibit the highest number of extreme values, followed by Chloramines, Trihalomethanes, Solids, Organic Carbon, Turbidity, and Conductivity. Although these values are far from the median, they should not be considered errors, as they likely represent normal variability in environmental water quality measurements.
#### Given the observed variability and the number of extreme values, it is not recommended to remove outliers at this stage. However, parameters with substantial outliers, such as Sulfate and pH, should be carefully considered in future modeling steps. Scaling or log-transformations may be applied when necessary to reduce the influence of extreme values on predictive models.

#### 6.2.2) Histplot distribution

In [ ]:
water_imputation.drop(columns='Potability').hist(figsize=(10,8),bins=30)
plt.suptitle('Distribution of Water Quality Variables')
plt.show()

In [ ]:
water_imputation.drop(columns='Potability').skew()

#### Skewness quantifies the asymmetry of a distribution, with positive values indicating a right skew and negative values indicating a left skew.
#### Histograms were used to assess the overall distribution of the parameters. Most variables exhibit approximately normal distributions, with slight positive skew observed in Solids (0.62) and Conductivity (0.26), while the rest of the parameters have near-zero skew, indicating symmetry around the mean. pH and Sulfate show a sharp peak around 7 and 333, respectively, indicating a concentration of samples within a narrow range.
#### These observations suggest that most variables can be modeled without transformation, while Solids and Conductivity may benefit from log-transformation in modeling to handle slight skewness. No action is required at this stage, but the distributions provide guidance for preprocessing decisions in future predictive analyses.

#### 6.2.3) Violin plot analysis

In [ ]:
plt.figure(figsize=(12,6))
sns.violinplot(data=water_imputation.drop(columns='Potability'))
plt.title('Violin Plot of Water Quality Variables')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
sns.violinplot(data=water_imputation[['ph','Chloramines','Organic_carbon','Turbidity']])
plt.title('Violin Plot of Small Range of Variables 1')
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
sns.violinplot(data=water_imputation[['Hardness','Sulfate','Conductivity','Trihalomethanes']])
plt.title('Violin Plot of Small Range of Variables 1')
plt.show()

In [ ]:
water_imputation.drop(columns='Potability').kurtosis()

#### Kurtosis measures the “peakedness” of the distribution, with values above 0 indicating a sharper peak than normal and values below 0 indicating flatter distributions. 
#### Violin plot analysis was used to evaluate data density and distribution shape across variables. The visual patterns are supported by kurtosis values, which measure the concentration of data around the central region. Higher kurtosis values, such as those observed in Sulfate (1.78) and pH (1.37), indicate distributions with sharper peaks and stronger concentration of values near the median. 
#### Variables such as Conductivity (-0.27) and Turbidity (-0.06) show negative kurtosis, indicating flatter distributions with less concentration around the central values. These results confirm that the variability observed in previous visualizations represents natural environmental dispersion rather than abnormal data patterns, supporting the decision to retain all observations for future modeling

### Conclusions

#### Overall, exploratory visualizations indicate that Sulfate, pH, and Hardness contain the highest presence of extreme values, while the remaining variables show moderate dispersion and generally symmetric distributions. Skewness and kurtosis analyses confirm that most parameters follow approximately normal distributions, with minor deviations observed in Solids and Conductivity.
#### Based on these findings, outliers will be retained because they likely represent realistic environmental variations. Future modeling stages may incorporate scaling or logarithmic transformations, particularly for Solids, Sulfate, and pH, to reduce the impact of extreme values and improve predictive model stability.

### 6.3) Outliers by target class

#### 6.3.1) Statistical comparison between classes

In [ ]:
water_imputation.groupby('Potability').mean()

#### Statistical comparison of feature means between potable and non-potable water samples shows only moderate differences across most variables. The absence of strong separation between class averages suggests that individual water quality parameters alone may not be sufficient predictors of potability.
#### Additionally, the similarity in mean values indicates that extreme values are not disproportionately influencing one specific class. This suggests that outliers are likely distributed across both groups rather than driving class differentiation.
#### From a modeling perspective, these results indicate that water potability may depend on multivariate interactions rather than single-feature thresholds, reinforcing the need for machine learning models capable of capturing non-linear relationships.

#### 6.3.2) Visual outliers by class

In [ ]:
for col in water_imputation.drop(columns="Potability"):
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Potability", y=col, data=water_imputation)
    plt.title(f"{col} vs Potability")
    plt.show()

#### Visual inspection of boxplots grouped by potability reveals substantial overlap in feature distributions between potable and non-potable water samples. Outliers are present in both classes across multiple variables, particularly in Sulfate, pH, and Hardness.
#### The presence of extreme values across both classes suggests that outliers alone do not serve as reliable indicators of water potability. Instead, these extreme observations appear to represent natural variability within water quality measurements.
#### These findings further support the assumption that classification will require models capable of identifying complex multi-variable patterns rather than relying on isolated extreme values.

#### 6.3.3) Outlier frequency comparison

In [ ]:
for col in water_imputation.drop(columns="Potability").columns:    
    print(f"\nVariable: {col}")
    for pot in [0,1]:
        subset = water_imputation[water_imputation["Potability"] == pot][col]
        Q1 = subset.quantile(0.25)
        Q3 = subset.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = subset[(subset < lower) | (subset > upper)]
        print(f"Potability {pot}: {len(outliers)} outliers")

#### Numerical outlier analysis by potability class confirms that extreme values are present in both potable and non-potable samples across most variables. No class shows systematic dominance in outlier occurrence.
#### This balanced distribution suggests that extreme observations are inherent to the dataset rather than being class-specific anomalies. Consequently, removing outliers aggressively could lead to loss of relevant information and reduce model generalization capacity.

### 6.4) Outlier detection using IQR method

In [ ]:
outlier_counts = {}
outlier_percentages = {}

n = len(water_imputation)

for col in water_imputation.drop(columns="Potability").columns:
    
    Q1 = water_imputation[col].quantile(0.25)
    Q3 = water_imputation[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = water_imputation[
        (water_imputation[col] < lower_bound) | 
        (water_imputation[col] > upper_bound)
    ]
    
    outlier_counts[col] = len(outliers)
    outlier_percentages[col] = (len(outliers) / n) * 100

In [ ]:
outlier_summary = pd.DataFrame({
    "Outlier Count": outlier_counts,
    "Outlier Percentage (%)": outlier_percentages
})

outlier_summary.sort_values(by="Outlier Count", ascending=False)

In [ ]:
outlier_summary.plot.bar()
plt.title("Number of Outliers per Variable")
plt.ylabel("Outlier Count")
plt.show()

#### The Interquartile Range (IQR) method was applied to formally identify outliers across all numerical water quality parameters. The results indicate that Sulfate and pH exhibit the highest number of extreme values, followed by Hardness and Chloramines.
#### From an environmental perspective, these findings are consistent with natural water variability, as parameters such as sulfate concentration and pH are influenced by geological composition, treatment processes, and potential contamination sources.
#### Despite the presence of outliers, their relative proportion within the dataset remains moderate, with the highest concentration representing less than 10% of total observations. This suggests that extreme values are unlikely to be data entry errors but rather reflect natural environmental fluctuations.
#### From a modeling standpoint, these results highlight the importance of carefully evaluating outlier treatment strategies, as removing them could potentially discard meaningful environmental information.

### 6.5) Outlier treatment strategy

#### 6.5.1) Possible outlier treatment techniques 

#### Several professional strategies exist to treat outliers:
#### 1) Remove Outliers – Eliminate extreme values (Pros: Reduces noise, can improve linear models. Cons: In environmental datasets, extreme values often reflect real conditions; removing them reduces representativeness and may introduce bias).
#### 2) Transform Outliers – Apply log, square root, or other transformations to reduce skewness (Pros: Makes distributions more normal; improves some model performance, Cons: Reduces physical interpretability; often unnecessary for non-linear models).
#### 3) Winsorization (Clipping) – Limit extreme values to a reasonable range without removing them (Pros: Maintains dataset size, reduces impact of extreme values, Cons: May mask real environmental events).
#### 4) Use Robust Machine Learning Models – Models inherently insensitive to outliers, e.g., Random Forest, Gradient Boosting, XGBoost, Decision Trees (Pros: Handle non-linear relationships and outliers without removing data).
#### In this step, the analysis of the dataset and environmental context guides the decision among these strategies.

#### 6.5.2) Decision based on environmental context 

#### Outliers are present in a relatively small proportion of observations across most variables (generally below 10%), suggesting that extreme values do not dominate the dataset structure. Variables showing higher extreme values, such as sulfate, pH, and hardness, likely reflect natural environmental variability rather than measurement errors.
#### Removing outliers would reduce meaningful information about real water quality conditions and could introduce artificial bias by eliminating valid environmental observations. Additionally, tree-based machine learning models such as Random Forest and Gradient Boosting are known to be robust to outliers and capable of capturing nonlinear relationships.
#### Therefore, all observations are retained in their original form, and no outlier removal or transformation is applied. Future modeling stages may explore scaling or logarithmic transformation for selected variables, such as solids or sulfate, depending on model-specific performance requirements. Retaining outliers ensures preservation of environmental variability and supports the development of realistic predictive models. models.

# SECTION 2: DATA PREPARATION & FEATURE ENGINEERING

## 7) Data Cleaning Decisions

### 7.1) Summary of Data Cleaning Actions

In [ ]:
water_imputation.isnull().sum()

In [ ]:
water_imputation.shape

#### Missing values were addressed during the preprocessing stage using appropriate imputation techniques, resulting in a complete dataset with no remaining null values. The final validation confirms that all variables contain valid observations, ensuring dataset integrity for further analysis.
#### Outlier analysis was performed using statistical and visualization techniques. Although extreme values were identified in some water quality parameters, these observations were retained due to their environmental relevance. The final dataset maintains all original observations, preserving natural variability and scientific representativeness while remaining suitable for machine learning modeling.

### 7.2) Final Dataset Quality Verification 

In [ ]:
water_imputation.duplicated().sum()

In [ ]:
water_imputation.describe()

In [ ]:
water_imputation.info()

#### Final dataset verification confirmed that no duplicate records are present, ensuring that each observation represents a unique water quality sample. Statistical summaries indicate that all variables fall within reasonable environmental ranges and maintain consistent distributions.
#### Additionally, all dataset features are stored using appropriate numerical data types, confirming compatibility with machine learning algorithms. These validation checks demonstrate that the dataset is structurally complete, internally consistent, and suitable for predictive modeling.

### 7.3) Final Remarks Before Modeling

#### The final dataset preserves the natural variability inherent to environmental water quality parameters. Extreme observations identified during exploratory analysis were retained, as they represent realistic environmental conditions rather than measurement errors. This approach ensures that the dataset maintains scientific validity and real-world representativeness.
#### Considering the dataset structure, absence of missing values, and preserved environmental variability, the data is fully prepared for machine learning modeling. The cleaned dataset provides a reliable foundation for training classification algorithms aimed at predicting water potability.

## 8) Feature Engineering & Feature Selection

### 8.1) Feature Scaling and Normalization

In [ ]:
# Compare minimum and maximum values of features
water_imputation.drop(columns="Potability").agg(['min', 'max']).T

In [ ]:
# Single boxplot to compare feature scales
plt.figure(figsize=(10,4))
sns.boxplot(data=water_imputation.drop(columns="Potability"))
plt.xticks(rotation=45)
plt.title("Comparison of Feature Scales")
plt.show()

#### Feature scale comparison indicates substantial differences in numerical magnitude across variables. Parameters such as Solids and Conductivity exhibit considerably larger ranges compared to variables like pH and Turbidity. These discrepancies reflect natural measurement scales rather than inconsistencies in the dataset (In case scaling is required for distance-based or gradient-sensitive algorithms, StandardScaler or MinMaxScaler can be applied to columns with large magnitudes, such as Solids or Conductivity).#### Machine learning algorithms that rely on distance calculations or gradient optimization, such as Logistic Regression, K-Nearest Neighbors, and Support Vector Machines, may be sensitive to feature scale. In contrast, tree-based algorithms including Random Forest and Gradient Boosting perform threshold-based splits and are therefore largely insensitive to differences in magnitude.
#### Given the environmental complexity of the dataset and the planned use of tree-based classification models, feature scaling is not applied at this stage. However, scaling remains a valid preprocessing option if scale-sensitive algorithms are later evaluated.d.

### 8.2) Feature Transformation 

In [ ]:
# Evaluate skewness of numerical features
skew_values = water_imputation.drop(columns="Potability").skew()
skew_values.sort_values(ascending=False)

In [ ]:
water_imputation.drop(columns='Potability').kurtosis().sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(4,2))
sns.histplot(water_imputation["Solids"], kde=True)
plt.title("Distribution of Solids")
plt.show()

plt.figure(figsize=(4,2))
sns.histplot(water_imputation["Conductivity"], kde=True)
plt.title("Distribution of Conductivity")
plt.show()

plt.figure(figsize=(4,2))
sns.histplot(water_imputation["ph"], kde=True)
plt.title("Distribution of pH")
plt.show()

plt.figure(figsize=(4,2))
sns.histplot(water_imputation["Sulfate"], kde=True)
plt.title("Distribution of Sulfate")
plt.show()

#### Skewness (-0.5 - 0.5)  approximately symmetrical distribution
#### Skewness (0.5 - 1)     moderately skewed
#### Skewness > 1 or < -1   highly skewd (possible candidate for transformation)

#### Distribution analysis reveals differences in concentration patterns across features. Variables such as pH and Sulfate exhibit pronounced central peaks, indicating strong clustering of observations around their mean values. This suggests positive kurtosis and high density in the central region.
#### Solids presents mild positive skewness, with a slightly extended right tail, while Conductivity displays a more uniform distribution. Overall, skewness levels remain within acceptable thresholds and do not indicate severe asymmetry (If transformations were required to reduce skewness and improve model performance in scale-sensitive algorithms, log (np.log1p), square root (np.sqrt), or Box-Cox (boxcox) transformations could be applied).
#### Although logarithmic or square root transformations could reduce mild skewness in variables such as Solids, transformation is not strictly required. Tree-based models are robust to moderate skewness and non-normal distributions. Therefore, no mathematical transformations are applied at this stage.

### 8.3) Categorical Encoding Considerations 

#### Categorical encoding is required when datasets contain non-numerical variables such as text labels or nominal categories. Machine learning algorithms operate on numerical inputs; therefore, categorical variables must be transformed into numerical representations before modeling.

#### Common encoding techniques include: Label Encoding (which assigns integer values to categories. This method is suitable for ordinal variables but may introduce artificial order in nominal categories), One-Hot Encoding (which creates binary indicator variables for each category. This method avoids introducing ordinal relationships and is widely used for nominal data), Target Encoding (which replaces categories with the mean target value. This technique is more advanced and may be useful in high-cardinality categorical features), Frequency Encoding (where categories are replaced by their occurrence frequency in the dataset).

#### In the current dataset, all predictor variables are already numerical and do not contain categorical attributes. The target variable (Potability) is encoded in binary format (0 = Not Potable, 1 = Potable), which is appropriate for classification modeling. Therefore, no categorical encoding is required at this stage.
#### If future environmental datasets include categorical features (e.g., water source, treatment type, season), appropriate encoding techniques will be incorporated into the preprocessing pipeline.

### 8.4) Feature Selection Analysis

#### 8.4.1) Correlation with target (Potability)

In [ ]:
# Calculate the correlation of each feature with the target variable
corr_target = water_imputation.corr()['Potability'].drop('Potability').sort_values(ascending=False)
print("Correlation with Potability:\n", corr_target)

#### This table shows the correlation of each feature with the target variable, Potability. Positive values ​​indicate that the higher feature values are associated with a higher probability of the water being potable, while negative values ​​indicate the opposite.
#### Correlation analysis shows that no individual feature has an extremely high correlation with Potability. Features like Solids, Chloramines and Trihalomethanes show modest positive correlations, while Organic_carbon shows a modest negative correlation, suggesting that potability is influenced by complex interactions rather than a single variable.

#### 8.4.2) Boxplots vs Potability

In [ ]:
# Boxplots by variable vs Potability
features = water_imputation.drop(columns="Potability").columns

plt.figure(figsize=(15,10))
for i, col in enumerate(features):
    plt.subplot(3,3,i+1)
    sns.boxplot(x='Potability', y=col, data=water_imputation)
    plt.title(f"{col} vs Potability")
plt.tight_layout()
plt.show()

#### This plot shows the distribution of each variable according to the Potability class. Most variables exhibit significant overlap between potable and non-potable water, indicating that no single feature can perfectly separate the classes.
#### However, features such as Solids and Sulfate show slight differences in the class means, suggesting they provide useful information for prediction. Other variables, display almost complete overlap, indicating lower individual relevance. 
#### Overall, the boxplot visualization confirms that while some features show distributional differences, substantial overlap exists, justifying the use of multi-feature models for accurate prediction.

#### 8.4.3) Groupby mean by class

In [ ]:
# Compare means by Potability class
grouped_means = water_imputation.groupby('Potability').mean()
print(grouped_means)

#### This table shows the mean value of each feature by potability class. 
#### Mean comparison by Potability confirms that some parameters (Solids, Sulfate) differ on average between potable and non-potable samples. While the differences are noticeable, distributions are overlapping, highlighting the need for models that can handle complex, multi-feature interactions.

#### 8.4.4) Mutual information (feature importance)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X = water_imputation.drop(columns='Potability')
y = water_imputation['Potability']

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print("Mutual Information Scores:\n", mi_series)

#### This table shows the mutual information of each feature with Potability. Higher values ​​indicate a greater capacity of the variable to provide information about the class. Hardness, Conductivity and Organic_carbon show the highest values (suggesting they are more informative for predictive modeling), while Chloramines and Trihalomethanes show values ​​close to zero.
#### This helps prioritize variables for modeling, but does not imply eliminating the others, since robust models like Random Forest can capture the combined information of all variables..

### 8.5) Multicollinearity Considerations 

#### 8.5.1) Correlation Matrix

In [ ]:
# Compute correlation matrix
corr_matrix = water_imputation.drop(columns="Potability").corr()

# Plot heatmap
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

#### The correlation matrix allows visualization of linear relationships between features. Most feature pairs exhibit low to moderate correlations, indicating limited redundancy across variables. No extreme correlations (e.g., > 0.8) are observed. This suggests potential interdependencies that may not be evident through simple pairwise correlation alone.

#### 8.5.2) Variance Inflation Factor (VIF)

In [ ]:
X = water_imputation.drop(columns="Potability")

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

vif_data = vif_data.sort_values(by="VIF", ascending=False)
print(vif_data)

#### VIF ≈ 1  → no collinearity
#### VIF 1–5  → moderate 
#### VIF 5–10  → concerning
#### VIF = 10  → serious
#### VIF = 20  → severe
#### VIF = 50  → extreme

#### Multicollinearity measures how much one variable can be explained by all the others together (it does not measure simple correlation).
#### Variables such as Sulfate, Hardness and Conductivity  exhibit high VIF values, indicating strong multicollinearity. This suggests that some predictors can be explained as linear combinations of others, reflecting interrelated environmental processes.
#### High multicollinearity would pose a significant issue for linear regression models, as it inflates coefficient variance, reduces interpretability, and may destabilize parameter estimates. However, tree-based algorithms such as Random Forest and Gradient Boosting are less sensitive to multicollinearity, since they rely on recursive partitioning rather than linear coefficient estimation.

#### 8.5.3) Model Considerations

#### Although strong multicollinearity is detected based on VIF analysis, no feature removal is performed at this stage. In environmental datasets, many water quality parameters are naturally interrelated (e.g., dissolved solids, conductivity, sulfate concentration), and removing variables may result in the loss of meaningful environmental information.
#### Given the intended use of tree-based classification models, which are inherently robust to multicollinearity, all features are retained for modeling. If linear or regression-based algorithms were to be implemented, dimensionality reduction techniques or regularization methods (e.g., Ridge or Lasso) would be considered.

### 8.6) Class Balance Evaluation

#### 0 = Not potable water
#### 1 = Potable water

#### 50/50  Perfect balance
#### 60/40  Slight imbalance
#### 70/30  Moderate
#### 80/20  Severe

In [ ]:
# Count class distribution
class_counts = water_imputation["Potability"].value_counts()
print(class_counts)

In [ ]:
# Class proportion
class_proportion = water_imputation["Potability"].value_counts(normalize=True)
print(class_proportion)

In [ ]:
sns.countplot(x="Potability", data=water_imputation, palette=["#1f77b4", "#ff7f0e"])
plt.title("Class Distribution of Potability")
plt.show()

#### If stronger imbalance were present, techniques such as SMOTE (it generates synthetic data for the minority class, small dataset), undersampling (it reduces the majority class, big dataset), or class weighting (it doesn't change data, it penalizes errors more in the minority class. Used in Logistic Regression or Random Forest) would be considered during the modeling phase. So, given the moderate distribution observed, no resampling strategy is applied at this stage. 
#### Class distribution analysis indicates that 60.99% of the observations correspond to non-potable water (class 0), while 39.01% correspond to potable water (class 1). This represents a mild class imbalance.e.
#### Although a mild class imbalance is observed, the distribution does not indicate a severe imbalance that would require resampling techniques at this stage.

### 8.7) Feature Engineering Summary & Modeling Preparation 

#### The feature engineering process evaluated potential preprocessing requirements including feature scaling, transformation, multicollinearity, and class balance considerations. Although substantial differences in feature magnitudes were identified, scaling was not applied at this stage due to the intended use of tree-based models, which are inherently robust to differences in numerical scale.
#### Distribution analysis indicated mild skewness in certain variables such as Solids and Conductivity; however, no severe asymmetry was detected that would require mathematical transformation. 
#### No categorical encoding was required, as all predictor variables were already represented in numerical format. The target variable (Potability) is appropriately encoded in binary form for classification modeling.
#### Feature selection analysis demonstrated that no single variable strongly predicts water potability independently. Instead, prediction is expected to rely on multivariate interactions among several parameters. 
#### Additionally, multicollinearity analysis revealed strong interdependencies among some environmental parameters based on VIF results. While this would be problematic for linear regression models, tree-based algorithms are less sensitive to multicollinearity and can effectively manage correlated predictors.
#### Class balance evaluation indicated a mild imbalance (approximately 61% non-potable and 39% potable samples), which does not necessitate resampling strategies at this stage.
#### Based on these findings, all features are retained in their original form. The dataset preserves environmental variability, maintains scientific integrity, and is fully prepared for train-test splitting and subsequent machine learning modeling.

## 9) Dataset Preparation For Modeling

### 9.1) Define Features and Target Variables

In [ ]:
# Define features (X) and target variable (y)
X = water_imputation.drop(columns="Potability")
y = water_imputation["Potability"]

# Verify shapes
print("X shape:", X.shape)
print("y shape:", y.shape)

#### The dataset was separated into predictor variables (X) and the target variable (y). All water quality parameters were included as features, while Potability was defined as the binary classification target.

### 9.2) Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Verify split proportions
print("Training set class distribution:\n", y_train.value_counts(normalize=True))
print("Testing set class distribution:\n", y_test.value_counts(normalize=True))

#### The dataset was divided into 80% training and 20% testing subsets
#### Stratified splitting was applied to preserve the original class distribution in both training and testing sets. This ensures that the mild class imbalance (61% vs 39%) is consistently represented, preventing biased evaluation.

### 9.3) Feature Scaling 

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit scaler only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply transformation to test data
X_test_scaled = scaler.transform(X_test)

#### Scaling = magnitude
#### Transformation = shape

#### It is important to distinguish between feature transformation and feature scaling. Feature transformation (e.g., logarithmic or square root transformation) is applied during the feature engineering stage (Step 8) when the objective is to modify the shape of a distribution, such as reducing skewness or stabilizing variance. These transformations alter the structural properties of the variable and are implemented prior to dataset splitting.
#### In contrast, feature scaling addresses differences in numerical magnitude across variables. Scaling does not modify distribution shape but standardizes feature ranges to prevent dominance of large-scale variables in scale-sensitive algorithms. Because scaling depends on dataset statistics (mean and standard deviation or min-max range), it must be applied after the train-test split and fitted exclusively on the training data to prevent data leakage.
#### Feature scaling was applied using StandardScaler exclusively for models sensitive to feature magnitude: Logistic Regression (can help), KNN (required), and SVM (highly recommended). These algorithms rely on distance calculations or gradient optimization, which can be affected by large numerical differences across variables. 
#### Tree-based models (Random Forest and Gradient Boosting) were trained on the original feature scale, as they are not sensitive to magnitude differences.

### 9.4) Class Imbalance Handling

In [ ]:
# Check class distribution in training set (counts and percentages)

class_counts_train = y_train.value_counts()
class_percent_train = y_train.value_counts(normalize=True)

print("Training set class counts:\n", class_counts_train)
print("\nTraining set class percentages:\n", class_percent_train)

#### It is important to distinguish between evaluating class imbalance and applying resampling techniques. Class imbalance assessment is performed during the dataset preparation stage; however, corrective techniques such as SMOTE, random undersampling, or class weighting must be applied only after the train-test split.
#### The class distribution in the training set remains moderately imbalanced (approximately 61% non-potable and 39% potable). Given the absence of severe imbalance, no resampling techniques such as SMOTE or undersampling are applied at this stage.
#### If future modeling results indicate bias toward the majority class, class weighting or synthetic resampling techniques may be considered.

### 9.5) Final Dataset Shape Verification

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

#### The dataset was divided into 80% training (2,620 observations) and 20% testing (656 observations) subsets. The training and testing feature matrices retain all nine predictor variables, while the target vectors correspond to the binary Potability classification. The alignment between feature and target dimensions confirms correct dataset partitioning and readiness for model development.

### 9.6) Dataset Ready for Modeling

#### The dataset was successfully separated into training and testing subsets using stratified sampling to preserve class distribution. All predictor variables remain in numerical format, and no missing values are present. Feature scaling was evaluated but deferred, given the planned use of tree-based models.
#### The dataset maintains environmental variability, class representation, and structural integrity. With properly defined training and testing sets, the data is fully prepared for machine learning model development and performance evaluation.

# SECTION 3: MODELING

## 10) Model Selection & Training

### 10.1) Logistic Regression - Linear Model (LR)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression model
log_model = LogisticRegression(max_iter=1000, random_state=42)

# Train model
log_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_log = log_model.predict(X_test_scaled)

#### Logistic Regression (Baseline) was implemented as a linear baseline classification model. This algorithm estimates the probability of class membership using a linear decision boundary. Because Logistic Regression relies on gradient optimization and is sensitive to feature magnitude, standardized features were used during training. The model serves as a reference to evaluate whether linear relationships are sufficient to predict water potability.

### 10.2) K-Nearest Neighbors - Distance-Based Model (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)

#### K-Nearest Neighbors (KNN) was implemented as a distance-based classification model. This algorithm classifies observations based on the majority class among the closest neighbors in feature space. Because KNN relies on distance calculations, feature scaling was required to prevent variables with larger magnitudes from dominating the model.

### 10.3) Support Vector Machine - Margin-Based Model (SVM)

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(probability=True, random_state=42)  #This model does not have probabilities enabled by default like the others (predict_proba() will be used further)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

#### Support Vector Machine (SVM) was implemented to evaluate a margin-based classifier capable of handling complex decision boundaries. SVM attempts to maximize the separation between classes using an optimal hyperplane. Since SVM is sensitive to feature scale, standardized features were used during training.

### 10.4) Random Forest - Tree-Based Ensemble Model (RF)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train model
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

#### Random Forest was implemented as a tree-based ensemble classifier. This algorithm builds multiple decision trees independently and aggregates their predictions to improve stability and reduce overfitting. Random Forest is robust to non-linear relationships, multicollinearity, outliers, and feature magnitude differences. Therefore, scaling was not required for this model. It is particularly suitable for environmental datasets characterized by complex interactions among variables.

### 10.5) Gradient Boosting - Tree-Based Ensemble Model (GB)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)

#### Gradient Boosting was implemented as an advanced tree-based ensemble method. Unlike Random Forest, which builds trees independently, Gradient Boosting constructs trees sequentially, where each new tree attempts to correct errors from previous models. This approach can improve predictive performance in complex datasets. Feature scaling was not required for this model.

### 10.6) Initial Model Comparison

#### 10.6.1) Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

# Store accuracies
accuracy_results = {
    "Logistic Regression": accuracy_score(y_test, y_pred_log),
    "KNN": accuracy_score(y_test, y_pred_knn),
    "SVM": accuracy_score(y_test, y_pred_svm),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
    "Gradient Boosting": accuracy_score(y_test, y_pred_gb)
}

# Convert to DataFrame and sort
accuracy_df = pd.DataFrame(
    accuracy_results.items(),
    columns=["Model", "Accuracy"]
).sort_values(by="Accuracy", ascending=False)

print(accuracy_df)

#### 0.50        = guess
#### 0.60 - 0.70 = good, moderate performance
#### 0.80+       = very good, strong
#### 0.90+       = excellent

#### Accuracy measures the proportion of correctly classified observations in the test dataset. It compares the predicted labels (y_pred) with the true labels (y_test) and calculates the percentage of exact matches. Since this is a binary classification problem (0 = non-potable, 1 = potable), predictions must match exactly to be considered correct.
#### In this case, SVM achieved the highest overall accuracy (67.07%), meaning that approximately 67 out of every 100 test observations were correctly classified. Random Forest and Gradient Boosting followed closely, while KNN and Logistic Regression showed slightly lower performance.

#### 10.6.2) Classification reports  

In [ ]:
from sklearn.metrics import classification_report

print("\n" + "="*50)
print("Logistic Regression Report:\n")
print(classification_report(y_test, y_pred_log))

print("\n" + "="*50)
print("KNN Report:\n")
print(classification_report(y_test, y_pred_knn))

print("\n" + "="*50)
print("SVM Report:\n")
print(classification_report(y_test, y_pred_svm))

print("\n" + "="*50)
print("Random Forest Report Report:\n")
print(classification_report(y_test, y_pred_rf))

print("\n" + "="*50)
print("Gradient Boosting Report:\n")
print(classification_report(y_test, y_pred_gb))
print("\n" + "="*50)

#### Precision measures the proportion of correctly predicted observations for a given class out of all predictions made for that class. It answers the question: When the model predicts a class, how often is it correct?
#### Recall measures the proportion of actual observations of a class that were correctly identified by the model. It answers: Out of all real instances of a class, how many were correctly detected?
#### F1-score is the harmonic mean of precision and recall. It provides a balanced measure when both false positives and false negatives are important.
#### Support represents the actual number of occurrences of each class in the test dataset.
#### Macro average computes the unweighted mean of the metrics across classes.
#### Weighted average computes the mean weighted by the number of samples in each class.

#### In this study, particular attention must be given to the potable water class (class 1), as misclassifying contaminated water as potable represents a serious public health risk. Therefore, evaluation should not rely solely on overall accuracy but also consider precision, recall, and F1-score for class 1.
#### The Logistic Regression model failed to correctly identify potable water samples. It predicted all observations as non-potable, resulting in 0 precision and recall for class 1. Although overall accuracy reached 0.61 due to correct classification of non-potable water, the model is unsuitable for practical potable water prediction.
#### KNN demonstrated moderate ability to identify potable water, correctly detecting 32% of actual potable samples. While precision reached 51%, the relatively low recall indicates that many potable samples were misclassified as non-potable.
#### SVM achieved the highest precision for potable water (70%), meaning that when the model predicted water as potable, it was correct most of the time. However, recall remained low (27%), indicating that many potable samples were not detected.
#### Random Forest provided the highest F1-score for potable water (0.41), indicating a better balance between precision and recall compared to other models. Although recall remains moderate, this model offers the most balanced performance for detecting potable water.
#### Gradient Boosting showed similar behavior to Random Forest but with slightly lower recall for potable water detection.
#### Although SVM achieved the highest overall accuracy, Random Forest demonstrated the best balance between precision and recall for potable water detection. Considering the environmental and public health implications, Random Forest appears to provide the most reliable classification performance among the evaluated models.

#### 10.6.3) Confusion matrices 

In [ ]:
from sklearn.metrics import confusion_matrix

print("\n" + "="*50)
print("Logistic Regression Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_log))

print("\n" + "="*50)
print("KNN Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_knn))

print("\n" + "="*50)
print("SVM Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_svm))

print("\n" + "="*50)
print("Random Forest Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_rf))

print("\n" + "="*50)
print("Gradient Boosting Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_gb))
print("\n" + "="*50)

#### [[TN  FP]
#### [FN  TP]]
#### TN → True Negative   - non-potable water correctly classified
#### TP → True Positive   - potable water correctly classified
#### FP → False Positive  - non-potable water incorrectly classified as potable (critical error)
#### FN → False Negative  - potable water incorectly classified as non-potable (error)

#### In water quality classification, false positives represent the most critical error, as contaminated water may be incorrectly labeled as safe for consumption.
#### The confusion matrices confirm the patterns observed in the classification reports. Logistic Regression failed to detect potable water entirely, while Random Forest and SVM demonstrated better balance between correctly identified potable and non-potable samples.

## 11) Model Evaluation

### 11.1) Confusion Matrix Analysis

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

models = [("SVM", y_pred_svm), ("Random Forest", y_pred_rf), ("Logistic Regression", y_pred_log), ("KNN", y_pred_knn), ("Gradient Boosting", y_pred_gb)]

fig, axes = plt.subplots(2, 3, figsize=(10,6))
axes = axes.flatten()

# Calculate Confusion Matrices
for i, (name, y_pred) in enumerate(models):
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[i], colorbar=False)
    axes[i].set_title(f"{name} Confusion Matrix")

# Turn off the last empty space
axes[len(models)].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
models = [("SVM", y_pred_svm, "Oranges"), ("Random Forest", y_pred_rf, "Greens"), ("Logistic Regression", y_pred_log, "Blues"),
    ("KNN", y_pred_knn, "Purples"), ("Gradient Boosting", y_pred_gb, "Reds")]

fig, axes = plt.subplots(2, 3, figsize=(10,6))
axes = axes.flatten()

for i, (name, y_pred, cmap) in enumerate(models):
    
    # Generate Confusion Matrices
    cm = confusion_matrix(y_test, y_pred)
    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=axes[i], cbar=False)
    
    axes[i].set_title(f"{name}\nAccuracy: {acc:.2f}")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

# Turn off empty spaces
for ax in axes[len(models):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

#### The confusion matrices provide a direct numerical comparison of classification performance across all evaluated models.
#### KNN achieved the highest number of correctly detected potable water samples (82 true positives), followed by Random Forest (77) and SVM (69). 
#### However, KNN also produced the highest number of contaminated water samples incorrectly classified as potable (79 false positives), indicating a significantly higher public health risk.
#### Logistic Regression generated the lowest number of false positives (0). Nevertheless, it completely failed to detect potable water (0 true positives), classifying all samples as non-potable. Although it avoids critical contamination errors, it lacks practical predictive capability.
#### Among the functional models, SVM produced the lowest number of false positives (29), followed by Gradient Boosting (35) and Random Forest (45). This suggests that SVM provides stronger control over critical contamination misclassification errors.
#### Random Forest presents a trade-off: it detects more potable samples (77 true positives) than SVM (69 true positives), but at the cost of increasing contamination risk (45 false positives vs 29 for SVM).
#### Gradient Boosting demonstrates intermediate behavior, with moderate potable detection (63 true positives) and controlled contamination misclassification (35 false positives).
#### Overall, the confusion matrix analysis highlights a structural trade-off between potable detection capability (true positives) and contamination misclassification risk (false positives). Model selection therefore depends on whether maximizing potable detection or minimizing critical contamination errors is prioritized in the environmental decision context.

### 11.2) Accuracy Discussion

In [ ]:
print("SVM Accuracy (Test):", accuracy_score(y_test, y_pred_svm))
print("RF Accuracy (Test):", accuracy_score(y_test, y_pred_rf))
print("GB Accuracy (Test):", accuracy_score(y_test, y_pred_gb))
print("KNN Accuracy (Test):", accuracy_score(y_test, y_pred_knn))
print("LR Accuracy (Test):", accuracy_score(y_test, y_pred_log))

#### Accuracy represents the proportion of correctly classified observations in the test dataset. For example, an accuracy of 0.67 indicates that 67% of the predictions match the actual values in the test set.
#### However, accuracy alone may be misleading in classification problems where class distribution is uneven or where certain types of errors carry greater consequences.
#### In this study, SVM achieved the highest overall accuracy (0.6707), followed closely by Random Forest (0.6585) and Gradient Boosting (0.6524). Although the numerical differences are small (1%), SVM appears slightly superior when considering total correct classification performance. KNN (0.6143) and Logistic Regression (0.6097) showed lower overall predictive accuracy.
#### Nevertheless, the confusion matrix analysis revealed that higher accuracy does not necessarily correspond to better potable water detection or lower critical misclassification rates. Therefore, accuracy should be interpreted alongside precision, recall, and false positive rates to ensure a meaningful environmental risk assessment.

### 11.3) Precision, Recall, F1-score Analysis

In [ ]:
models = {"SVM": y_pred_svm, "RF": y_pred_rf, "LR": y_pred_log, "KNN": y_pred_knn, "GB": y_pred_gb}
metrics_data = []

for name, y_pred in models.items():
    
    report = classification_report(y_test, y_pred, output_dict=True)  
    metrics_data.append({
        "Model": name,
        "Precision": report["1"]["precision"],
        "Recall": report["1"]["recall"],
        "F1-Score": report["1"]["f1-score"]
    })

df_metrics = pd.DataFrame(metrics_data).set_index("Model")

In [ ]:
ax = df_metrics.plot(kind="bar", figsize=(10,6), width=0.8)

plt.title("Precision, Recall and F1-Score (Class 1 - Potable)")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=45)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()

#### Precision indicates how reliable the model is when predicting a specific class. In this context, it measures how many samples predicted as potable were actually potable.
#### Recall indicates how many actual samples of a class were correctly identified. For potable water (class 1), recall measures how many truly potable samples were successfully detected by the model.
#### F1-score provides a balanced metric between precision and recall. It is particularly useful when there is a trade-off between detecting more potable samples and minimizing critical misclassification errors.
#### Although SVM achieved the highest overall accuracy in the previous evaluation, Random Forest demonstrates the best balance between precision and recall for potable water detection (F1-score = 0.41 for class 1). SVM and Gradient Boosting show slightly lower but comparable F1-scores, while KNN presents moderate performance.#### Logistic Regression is unsuitable for this application because it fails entirely to detect potable samples (recall = 0 for class 1), resulting in an F1-score of 0.
#### From an environmental and public health perspective, recall for potable water remains a critical metric, since failing to correctly identify safe water reduces the model’s practical usability. However, none of the evaluated models achieve high recall for potable water, suggesting that further model optimization, threshold tuning, or class balancing strategies may be required.

### 11.4) Receiver Operating Characteristic (ROC Curve) & Area Under de Curve (AUC Score)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

models = [("SVM", svm_model, X_test), ("Random Forest", rf_model, X_test), ("Logistic Regression", log_model, X_test_scaled),
    ("KNN", knn_model, X_test_scaled), ("Gradient Boosting", gb_model, X_test)] # SVM (need probability=True in the model)

plt.figure(figsize=(8,6))

for name, model, X_data in models:
    y_prob = model.predict_proba(X_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob) 
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.2f})")

# Random baseline
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()

#### AUC interpretation
#### 0.5 → random classification
#### 0.7 → acceptable discrimination
#### 0.8 → good discrimination
#### 0.9 → excellent discrimination
#### 1.0 → perfect separation

#### The ROC curve evaluates the global discriminative ability of each model across all possible classification thresholds. While accuracy depends on a fixed threshold (commonly 0.5), the ROC curve assesses how well the model separates classes regardless of the chosen cutoff. 
#### In this study, Random Forest achieved the highest AUC (0.64), followed closely by Gradient Boosting (0.63), indicating moderate class separation capability.These values suggest that both models possess some ability to rank potable samples higher than non-potable ones, although performance remains far from optimal.
#### SVM obtained an AUC of 0.50, suggesting performance equivalent to random classification in terms of probability ranking.
#### Although SVM achieved slightly higher accuracy and fewer false positives, its inability to effectively separate classes across thresholds indicates limited structural discrimination capacity. This indicates that, although SVM achieved slightly higher accuracy and fewer false positives under the fixed threshold evaluation, it lacks structural discrimination capacity when evaluated across all thresholds.
#### Random Forest, despite producing slightly more false positives in the confucion matrix analysis, demonstrates stronger global separation between potable and non-potable samples.
#### Therefore, when considering both fixed-threshold performance (accuracy and confusion matrix) and global discrimination ability (AUC), Random Forest emerges as the more robust and reliable model for this dataset.

### 11.5) Model Comparison & Integrated Evaluation

In [ ]:
models = {"SVM": (svm_model, y_pred_svm, X_test), "Random Forest": (rf_model, y_pred_rf, X_test), "Gradient Boosting": (gb_model, y_pred_gb, X_test),
    "KNN": (knn_model, y_pred_knn, X_test_scaled), "Logistic Regression": (log_model, y_pred_log, X_test_scaled)}

comparison_data = []

for name, (model, y_pred, X_data) in models.items(): 
    # Classification report
    report = classification_report(y_test, y_pred, output_dict=True)   
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    # Accuracy
    acc = accuracy_score(y_test, y_pred)
    # AUC
    y_prob = model.predict_proba(X_data)[:, 1]
    auc_score = roc_auc_score(y_test, y_prob)
    
    comparison_data.append({"Model": name, "Accuracy": acc, "AUC": auc_score, "True Positives (TP)": tp, "False Positives (FP)": fp,
        "Recall (Class 1)": report["1"]["recall"], "F1-Score (Class 1)": report["1"]["f1-score"]})

comparison_table = pd.DataFrame(comparison_data).sort_values(by="F1-Score (Class 1)", ascending=False)
comparison_table

In [ ]:
summary_data = {"Metric": ["Highest Accuracy", "Highest AUC", "Highest True Positives (TP)", "Lowest False Positives (FP)",
                "Lowest False Positives (Functional Model)", "Best Recall (Class 1)", "Best F1-Score (Class 1)"],
                "Best Model": ["SVM", "Random Forest", "KNN", "Logistic Regression", "SVM", "KNN", "Random Forest"], 
                "Value": [0.67, 0.64, 82, 0, 29, 0.32, 0.41]}
df_summary = pd.DataFrame(summary_data)
df_summary

#### The integrated comparison table highlights clear performance differences across evaluation criteria.
#### SVM achieved the highest overall accuracy (0.67) and the lowest number of false positives among functional models (29), indicating strong control over contamination misclassification under a fixed threshold.
#### Random Forest achieved the highest AUC (0.64), suggesting superior global class separation capability. It also obtained the highest F1-score for potable water detection (0.41), indicating the best balance between precision and recall for class 1.
#### KNN demonstrated the highest number of correctly detected potable samples (82 true positives) and the highest recall for potable water (0.32). However, it also produced the highest number of false positives (79), representing a substantial contamination misclassification risk.
#### Gradient Boosting showed consistent intermediate performance across most metrics, with moderate AUC (0.63) and controlled false positive rates (35).
#### Logistic Regression failed to detect potable water entirely and is therefore unsuitable for this application despite generating zero false positives.
#### When integrating all performance indicators — fixed-threshold accuracy, critical error control (false positives), potable detection capability (recall and true positives), F1-score balance, and global discrimination ability (AUC) — Random Forest demonstrates the most structurally balanced and robust performance for this dataset.
#### Although SVM performs well under fixed-threshold evaluation, its AUC of 0.50 suggests limited discrimination capability beyond the selected threshold.
#### Therefore, Random Forest is selected as the most appropriate model for further validation and generalization assessment.

### 11.6 Overfitting Assessment

In [ ]:
models_train_test = {"Logistic Regression": (log_model, X_train_scaled, X_test_scaled), "SVM": (svm_model, X_train_scaled, X_test_scaled), 
                     "KNN": (knn_model, X_train_scaled, X_test_scaled), "Random Forest": (rf_model, X_train, X_test), 
                     "Gradient Boosting": (gb_model, X_train, X_test)}

results = []

for name, (model, X_tr, X_te) in models_train_test.items():
    train_acc = model.score(X_tr, y_train)
    test_acc = model.score(X_te, y_test)
    results.append({"Model": name, "Train Accuracy": train_acc, "Test Accuracy": test_acc, "Gap (Train - Test)": train_acc - test_acc})

df_train_test = pd.DataFrame(results).set_index("Model").round(4)
df_train_test

#### 0.70 → good (depends on the problem)
#### 0.60–0.70 → moderate
#### < 0.60 → low
#### ≈ 0.50 → almost random in binary

#### Train ≈ Test (both reasonably high) -> The model learns generalizable patterns and does not memorize the training data. This represents good generalization and a well-fitted model.
#### Train = Test low (both reasonably low) -> The model is too simple and fails to capture meaningful patterns (underfitting).
#### Train >> Test -> The model memorizes the training data (overfitting). The larger the gap between training and testing performance, the more severe the overfitting.

#### To evaluate generalization capacity, training and testing accuracy were compared across all evaluated models.
#### Logistic Regression shows nearly identical training (0.6099) and testing accuracy (0.6098), indicating stable but limited predictive capacity. The low performance in both datasets suggests mild underfitting rather than overfitting.
#### SVM presents good generalization behavior, with training accuracy of 0.7382 and testing accuracy of 0.6707 (gap ≈ 0.067). The relatively small difference indicates acceptable stability and limited overfitting.
#### Gradient Boosting demonstrates moderate generalization with a training accuracy of 0.7420 and testing accuracy of 0.6524 (gap ≈ 0.089). Although some overfitting is present, it remains within acceptable limits.
#### KNN exhibits stronger overfitting tendencies, with a training accuracy of 0.7614 and testing accuracy of 0.6143 (gap ≈ 0.147). The larger discrepancy suggests that the model captures training patterns more aggressively, reducing generalization performance.
#### Random Forest shows severe overfitting, achieving perfect training accuracy (1.00) but considerably lower testing accuracy (0.6585). The substantial gap (≈ 0.34) indicates that the model memorizes the training data rather than learning generalizable patterns.
#### Overall, SVM and Gradient Boosting demonstrate the most stable generalization behavior, while Random Forest requires hyperparameter tuning to reduce model complexity and improve generalization capacity.

### 11.7 Final Model Selection

#### GB -> intermediate balance (intermediate candidate)
#### RF -> better AUC but worse generalization (powerful but with high overfitting)
#### SVM -> better stability but low AUC (stable but structurally weak)

#### Based on the integrated evaluation of accuracy, confusion matrices, AUC performance, and overfitting assessment, model selection must balance predictive strength and generalization stability.
#### Random Forest achieved the highest AUC but exhibited severe overfitting, indicating memorization of training data. While powerful, its generalization stability is limited without further regularization.
#### SVM demonstrated strong stability and the highest fixed-threshold accuracy. However, its AUC of 0.50 suggests weak structural discrimination capacity, limiting its robustness across classification thresholds.
#### Gradient Boosting presents the most balanced performance profile. It achieves competitive AUC (0.63), moderate overfitting, and stable generalization behavior without extreme memorization effects.
#### Therefore, Gradient Boosting is selected as the primary candidate for further hyperparameter optimization. Random Forest remains a secondary candidate due to its strong discriminative potential if properly regularized. SVM may be considered for exploratory tuning but is not prioritized given its limited AUC performance.

## 12) Model Optimization

### 12.1) Hyperparameter Tuning (Gradient Boosting)

#### 12.1.1) Hyperparameter selection strategy

#### A grid search strategy was implemented to optimize Gradient Boosting performance. The parameter ranges were selected to balance model complexity and generalization capacity.
#### Five-fold cross-validation (cv=5) was applied to ensure robust evaluation across different data splits. The scoring metric was set to F1-score to prioritize balanced performance for potable water detection.
#### n_estimators [100, 200]: controls the number of boosting stages. Higher values increase model capacity but may increase overfitting. 100–200 was selected as a moderate search range.
#### learning_rate [0.01, 0.05, 0.1]: determines how strongly each tree contributes. Lower learning rates improve stability but require more trees. Tested moderate values to balance convergence and generalization.
#### max_depth [3, 4, 5]: limits tree complexity. Smaller depth reduces overfitting. Values kept relatively shallow to control model variance.
#### min_samples_split / min_samples_leaf: prevent trees from splitting too aggressively. Help regularize the model and improve generalization.

#### 12.1.2) Best hyperparameters

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

# Define parameter grid
param_grid_gb = {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [3, 4, 5], 
                 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]}

gb = GradientBoostingClassifier(random_state=42)
grid_gb = GridSearchCV(estimator=gb, param_grid=param_grid_gb, cv=5, scoring='f1', n_jobs=-1)
grid_gb.fit(X_train, y_train)

# Best model
best_gb = grid_gb.best_estimator_

print("Best GB Parameters:", grid_gb.best_params_)

#### The optimized configuration increased the number of estimators while maintaining moderate tree depth.
#### This suggests that model performance benefits from a higher ensemble size rather than deeper trees.exity.


#### 12.1.3) Optimized performance

In [ ]:
y_pred_gb_opt = best_gb.predict(X_test)
print("Optimized GB Accuracy:", accuracy_score(y_test, y_pred_gb_opt))

In [ ]:
y_pred_gb_opt = best_gb.predict(X_test)
print(classification_report(y_test, y_pred_gb_opt))

In [ ]:
y_prob_gb_opt = best_gb.predict_proba(X_test)[:,1]
auc_gb_opt = roc_auc_score(y_test, y_prob_gb_opt)

print("Optimized GB AUC:", auc_gb_opt)

In [ ]:
# BASE MODEL METRICS 

# Accuracy base
acc_base = accuracy_score(y_test, y_pred_gb)
# Classification report base
report_base = classification_report(y_test, y_pred_gb, output_dict=True)
precision_base = report_base["1"]["precision"]
recall_base = report_base["1"]["recall"]
f1_base = report_base["1"]["f1-score"]
# AUC base
y_prob_base = gb_model.predict_proba(X_test)[:,1]
auc_base = roc_auc_score(y_test, y_prob_base)

# OPTIMIZED MODEL METRICS

# Accuracy optimized
acc_opt = accuracy_score(y_test, y_pred_gb_opt)
# Classification report optimized
report_opt = classification_report(y_test, y_pred_gb_opt, output_dict=True)
precision_opt = report_opt["1"]["precision"]
recall_opt = report_opt["1"]["recall"]
f1_opt = report_opt["1"]["f1-score"]
# AUC optimized
y_prob_opt = best_gb.predict_proba(X_test)[:,1]
auc_opt = roc_auc_score(y_test, y_prob_opt)

# CREATE COMPARISON TABLE 

df_gb_comparison = pd.DataFrame({
"Metric": ["Accuracy", "Precision (Class 1)", "Recall (Class 1)", "F1-score (Class 1)", "AUC"],
"Base Model": [acc_base, precision_base, recall_base, f1_base, auc_base],
"Optimized Model": [acc_opt, precision_opt, recall_opt, f1_opt, auc_opt]})

df_gb_comparison["Improvement"] = df_gb_comparison["Optimized Model"] - df_gb_comparison["Base Model"]
df_gb_comparison

#### The optimized Gradient Boosting model demonstrates a meaningful improvement in recall and F1-score for potable water detection, indicating enhanced sensitivity to class 1 samples.
#### Although precision decreased slightly, the increase in recall compensates for this reduction, resulting in improved balanced performance.
#### The small increase in AUC suggests stable global discrimination capacity.
#### Overall, the optimization led to incremental but relevant improvements without compromising structural stability.

#### 12.1.4) Generalization assessment (train vs test)

In [ ]:
# BASE
train_acc_base_gb = gb_model.score(X_train, y_train)
test_acc_base_gb = gb_model.score(X_test, y_test)

# OPTIMIZED
train_acc_opt_gb = best_gb.score(X_train, y_train)
test_acc_opt_gb = best_gb.score(X_test, y_test)

# Table
df_gb_generalization = pd.DataFrame({"Version": ["Base", "Optimized"], "Train Accuracy": [train_acc_base_gb, train_acc_opt_gb],
                                     "Test Accuracy": [test_acc_base_gb, test_acc_opt_gb]})

df_gb_generalization["Gap (Train - Test)"] = (df_gb_generalization["Train Accuracy"] - df_gb_generalization["Test Accuracy"])
df_gb_generalization

#### The optimized Gradient Boosting model slightly improved test accuracy (0.652 → 0.659). However, training accuracy increased substantially (0.742 → 0.956), resulting in a significantly larger generalization gap (0.089 → 0.297).
#### This indicates that although predictive performance improved marginally, the optimized model exhibits stronger overfitting behavior. The model has increased its capacity to memorize training data without achieving proportional improvement in unseen data performance.
#### Therefore, optimization improved discrimination metrics but reduced generalization stability.

### 12.2) Hyperparameter Tuning (Random Forest)

#### 12.2.1) Hyperparameter selection strategy

#### A Grid Search strategy was implemented to reduce the strong overfitting observed in the baseline Random Forest model.
#### The parameter grid focused on limiting tree depth, increasing minimum sample constraints, and controlling feature selection at each split.
#### Five-fold cross-validation (cv=5) was applied to ensure stable performance estimation.The scoring metric was set to F1-score to prioritize balanced detection of potable water samples.
#### n_estimators [100, 200]: a moderate ensemble size was selected to test whether additional trees improve stability without substantially increasing variance.
#### max_depth [5, 10, 15, None]: the inclusion of restricted tree depths aimed to reduce model variance and prevent memorization of training data. However, the optimal configuration retained max_depth=None, indicating preference for unrestricted tree growth. This result suggests that the dataset benefits from highly flexible tree structures, although this flexibility increases overfitting risk.
#### min_samples_split [2, 5, 10]: increasing the minimum samples required for splitting was intended to reduce excessive branching and improve generalization. 
#### min_samples_leaf [1, 2, 4]: larger leaf sizes were tested to smooth decision boundaries and reduce model variance. 
#### max_features [‘sqrt’, ‘log2’]: feature subset restriction was evaluated to increase tree diversity and potentially reduce correlation among trees. 

#### 12.2.2) Best hyperparameters

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Define parameter grid
param_grid_rf = {'n_estimators': [100, 200], 'max_depth': [5, 10, 15, None], 
                 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4], 'max_features': ['sqrt', 'log2']}

rf = RandomForestClassifier(random_state=42)
grid_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train, y_train)

# Best model
best_rf = grid_rf.best_estimator_

print("Best RF Parameters:", grid_rf.best_params_)

#### The Grid Search selected a configuration that maintains full tree depth, moderate ensemble size, and standard feature subsampling.
#### The chosen parameters indicate that the model benefits from flexible tree structures while maintaining controlled feature selection.


#### 12.2.3) Optimized performance

In [ ]:
y_pred_rf_opt = best_rf.predict(X_test)
print("Optimized RF Accuracy:", accuracy_score(y_test, y_pred_rf_opt))

In [ ]:
y_pred_rf_opt = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_rf_opt))

In [ ]:
y_prob_rf_opt = best_rf.predict_proba(X_test)[:,1]
auc_rf_opt = roc_auc_score(y_test, y_prob_rf_opt)

print("Optimized RF AUC:", auc_rf_opt)

In [ ]:
# BASE MODEL METRICS

# Accuracy base
acc_base_rf = accuracy_score(y_test, y_pred_rf)
# Classification report base
report_base_rf = classification_report(y_test, y_pred_rf, output_dict=True)
precision_base_rf = report_base_rf["1"]["precision"]
recall_base_rf = report_base_rf["1"]["recall"]
f1_base_rf = report_base_rf["1"]["f1-score"]
# AUC base
y_prob_base_rf = rf_model.predict_proba(X_test)[:, 1]
auc_base_rf = roc_auc_score(y_test, y_prob_base_rf)

# OPTIMIZED MODEL METRICS

# Accuracy optimized
acc_opt_rf = accuracy_score(y_test, y_pred_rf_opt)
# Classification report optimized
report_opt_rf = classification_report(y_test, y_pred_rf_opt, output_dict=True)
precision_opt_rf = report_opt_rf["1"]["precision"]
recall_opt_rf = report_opt_rf["1"]["recall"]
f1_opt_rf = report_opt_rf["1"]["f1-score"]
# AUC optimized
y_prob_opt_rf = best_rf.predict_proba(X_test)[:, 1]
auc_opt_rf = roc_auc_score(y_test, y_prob_opt_rf)

# CREATE COMPARISON TABLE

df_rf_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (Class 1)", "Recall (Class 1)", "F1-score (Class 1)", "AUC"],
    "Base Model": [acc_base_rf, precision_base_rf, recall_base_rf, f1_base_rf, auc_base_rf],
    "Optimized Model": [acc_opt_rf, precision_opt_rf, recall_opt_rf, f1_opt_rf, auc_opt_rf]})

df_rf_comparison["Improvement"] = (df_rf_comparison["Optimized Model"] - df_rf_comparison["Base Model"])
df_rf_comparison

#### The optimized Random Forest model shows consistent improvements across all evaluated metrics.
#### Accuracy, precision, recall, F1-score, and AUC all increased compared to the baseline configuration.
#### Although the improvements are moderate, they indicate that hyperparameter tuning contributed to enhanced classification performance.

#### 12.2.4) Generalization assessment (train vs test)

In [ ]:
# BASE
train_acc_base_rf = rf_model.score(X_train, y_train)
test_acc_base_rf = rf_model.score(X_test, y_test)

# OPTIMIZED
train_acc_opt_rf = best_rf.score(X_train, y_train)
test_acc_opt_rf = best_rf.score(X_test, y_test)

# Table
df_rf_generalization = pd.DataFrame({"Version": ["Base", "Optimized"], "Train Accuracy": [train_acc_base_rf, train_acc_opt_rf],
                                     "Test Accuracy": [test_acc_base_rf, test_acc_opt_rf]})

df_rf_generalization["Gap (Train - Test)"] = (df_rf_generalization["Train Accuracy"] - df_rf_generalization["Test Accuracy"])
df_rf_generalization

#### The optimized Random Forest model slightly improved test accuracy (0.6585 → 0.6662) while maintaining near-perfect training accuracy (≈1.0). The generalization gap decreased marginally (0.341 → 0.333), but remains substantially high.
#### This indicates that hyperparameter tuning produced incremental performance gains without effectively reducing structural overfitting. The model continues to exhibit strong memorization of training data.

### 12.3) Hyperparameter Tuning (Support Vector Machine)

#### 12.3.1) Hyperparameter selection strategy

#### A Grid Search strategy was implemented to optimize Support Vector Machine performance. Given that SVM performance is highly sensitive to the regularization parameter and kernel configuration, the parameter grid focused on controlling model complexity and margin flexibility.
#### Five-fold cross-validation (cv=5) was applied to ensure robust evaluation across different data splits. The scoring metric was set to F1-score in order to prioritize balanced performance for potable water detection.
#### C: controls the strength of regularization. Lower values increase regularization (simpler decision boundary), while higher values allow the model to fit the data more closely.
#### gamma: defines the influence of individual training samples in the RBF kernel. Smaller values produce smoother decision boundaries; larger values increase model flexibility.
#### kernel: RBF kernel was selected as it can capture nonlinear relationships between environmental variables.
#### The selected ranges aim to balance bias and variance while controlling potential overfitting.

#### 12.3.2) Best hyperparameters

In [ ]:
from sklearn.svm import SVC

# Define parameter grid
param_grid_svm = {'C': [0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1], 'kernel': ['rbf']}

svm = SVC(probability=True, random_state=42)
grid_svm = GridSearchCV(estimator=svm, param_grid=param_grid_svm, cv=5, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train_scaled, y_train)

# Best model
best_svm = grid_svm.best_estimator_

print("Best SVM Parameters:", grid_svm.best_params_)

#### The optimized configuration selected a non-linear RBF kernel with a high regularization parameter (C = 100) and moderate gamma (0.1). This indicates that the model benefits from increased flexibility while maintaining controlled influence of individual training samples.

#### 12.3.3) Optimized performance

In [ ]:
y_pred_svm_opt = best_svm.predict(X_test_scaled)
print("Optimized SVM Accuracy:", accuracy_score(y_test, y_pred_svm_opt))

In [ ]:
y_pred_svm_opt = best_svm.predict(X_test_scaled)
print(classification_report(y_test, y_pred_svm_opt))

In [ ]:
y_prob_svm_opt = best_svm.predict_proba(X_test_scaled)[:,1]
auc_svm_opt = roc_auc_score(y_test, y_prob_svm_opt)

print("Optimized SVM AUC:", auc_svm_opt)

In [ ]:
# BASE MODEL METRICS

# Accuracy base
acc_base_svm = accuracy_score(y_test, y_pred_svm)
# Classification report base
report_base_svm = classification_report(y_test, y_pred_svm, output_dict=True)
precision_base_svm = report_base_svm["1"]["precision"]
recall_base_svm = report_base_svm["1"]["recall"]
f1_base_svm = report_base_svm["1"]["f1-score"]
# AUC base
y_prob_base_svm = svm_model.predict_proba(X_test_scaled)[:,1]
auc_base_svm = roc_auc_score(y_test, y_prob_base_svm)

# OPTIMIZED METRICS

# Accuracy optimized
acc_opt_svm = accuracy_score(y_test, y_pred_svm_opt)
# Classification report optimized
report_opt_svm = classification_report(y_test, y_pred_svm_opt, output_dict=True)
precision_opt_svm = report_opt_svm["1"]["precision"]
recall_opt_svm = report_opt_svm["1"]["recall"]
f1_opt_svm = report_opt_svm["1"]["f1-score"]
# AUC optimized
y_prob_opt_svm = best_svm.predict_proba(X_test_scaled)[:, 1]
auc_opt_svm = roc_auc_score(y_test, y_prob_opt_svm)

# CREATE COMPARISON TABLE

df_svm_comparison = pd.DataFrame({"Metric": ["Accuracy", "Precision (Class 1)", "Recall (Class 1)", "F1-score (Class 1)", "AUC"], 
                                  "Base Model": [acc_base_svm, precision_base_svm, recall_base_svm, f1_base_svm, auc_base_svm],
                                  "Optimized Model": [acc_opt_svm, precision_opt_svm, recall_opt_svm, f1_opt_svm, auc_opt_svm]})

df_svm_comparison["Improvement"] = (df_svm_comparison["Optimized Model"] - df_svm_comparison["Base Model"])
df_svm_comparison

#### The optimized SVM model shows a trade-off in performance. While recall for class 1 (potable water) increased, indicating improved sensitivity in detecting potable samples, overall accuracy and precision decreased.
#### The reduction in precision suggests a higher number of false positives after optimization. Although the F1-score shows a slight improvement, the decrease in AUC indicates weaker overall discrimination capacity.
#### Overall, the optimization did not lead to a consistent improvement in global performance and may have introduced reduced model stability.ty.


#### 12.3.4) Generalization assessment (train vs test)


In [ ]:
# BASE
train_acc_base_svm = svm_model.score(X_train_scaled, y_train)
test_acc_base_svm = svm_model.score(X_test_scaled, y_test)

# OPTIMIZED
train_acc_opt_svm = best_svm.score(X_train_scaled, y_train)
test_acc_opt_svm = best_svm.score(X_test_scaled, y_test)

#Table
df_svm_generalization = pd.DataFrame({"Version": ["Base", "Optimized"], "Train Accuracy": [train_acc_base_svm, train_acc_opt_svm], 
                                      "Test Accuracy": [test_acc_base_svm, test_acc_opt_svm]})

df_svm_generalization["Gap (Train - Test)"] = (df_svm_generalization["Train Accuracy"] - df_svm_generalization["Test Accuracy"])
df_svm_generalization

#### The base SVM model shows moderate generalization, with a small gap between training and testing accuracy (0.738 vs 0.671). This indicates relatively stable learning without severe overfitting.
#### However, after optimization, training accuracy increased substantially (0.906), while test accuracy decreased notably (0.588). The generalization gap expanded significantly from 0.067 to 0.318.
#### This behavior suggests that the optimized SVM model suffers from strong overfitting. The model appears to memorize the training data rather than generalizing effectively to unseen samples.
#### Therefore, despite the hyperparameter tuning process, the optimized configuration reduces model stability and does not improve real-world predictive performance.

### 12.4) Re-evaluation of Optimized Models

In [ ]:
# FINAL COMPARISON (Optimized Models Only)

df_final_comparison = pd.DataFrame({"Model": ["Gradient Boosting", "Random Forest", "SVM"],

"Accuracy": [acc_opt, acc_opt_rf, acc_opt_svm],
"Precision (Class 1)": [precision_opt, precision_opt_rf, precision_opt_svm],
"Recall (Class 1)": [recall_opt, recall_opt_rf,recall_opt_svm],
"F1-score (Class 1)": [f1_opt, f1_opt_rf, f1_opt_svm],
"AUC": [auc_opt, auc_opt_rf, auc_opt_svm],
"Generalization Gap": [train_acc_opt_gb - test_acc_opt_gb, train_acc_opt_rf - test_acc_opt_rf, train_acc_opt_svm - test_acc_opt_svm]})

df_final_comparison

#### The table summarizes the performance of the optimized models after hyperparameter tuning.
#### Random Forest achieved the highest accuracy (0.666), while SVM obtained the highest recall for potable water detection (0.367). However, SVM showed weaker overall discrimination (lower AUC) and reduced stability.
#### Gradient Boosting achieved the highest F1-score (0.434), indicating the best balance between precision and recall. Additionally, it presented the smallest generalization gap, suggesting better stability compared to the other models.
#### Considering overall performance, balance between metrics, and generalization capacity, Gradient Boosting emerges as the most robust and reliable model for potable water classification in this study.udy.



### 12.5) Optimized Model Selection

#### Based on the comparative evaluation of the optimized models, Gradient Boosting is selected as the final model for potable water classification.
#### Although Random Forest achieved slightly higher accuracy, and SVM obtained the highest recall for class 1, Gradient Boosting demonstrated the best overall balance between performance metrics. It achieved the highest F1-score, indicating a strong equilibrium between precision and recall, which is critical for reliable potable water detection.
#### Furthermore, Gradient Boosting presented the smallest generalization gap between training and testing accuracy, suggesting better model stability and reduced overfitting compared to the other models.
#### Considering classification balance, discrimination capacity (AUC), and generalization performance, Gradient Boosting provides the most robust and reliable solution for this dataset.

## 13) Model Interpretation & Environmental Insights

### 13.1) Feature Importance Analysis (Gradient Boosting)

In [ ]:
feature_importance_df = pd.DataFrame({"Feature": X_train.columns, "Importance": best_gb.feature_importances_})
feature_importance_df = feature_importance_df.sort_values(by="Importance", ascending=False)
feature_importance_df

In [ ]:
plt.figure(figsize=(8,6))
plt.barh(feature_importance_df["Feature"], feature_importance_df["Importance"])

plt.xlabel("Importance Score")
plt.title("Feature Importance - Gradient Boosting")
plt.gca().invert_yaxis()
plt.show()

#### The feature importance analysis shows that pH is the most influential variable in the optimized Gradient Boosting model, followed by sulfate and chloramines.
#### Feature importance represents how much each variable contributes to the model’s decision-making process when classifying water as potable or non-potable. Higher importance values indicate that the variable played a stronger role in separating the two classes during tree construction.
#### The relatively balanced distribution of importance across several variables suggests that the model does not rely on a single parameter but instead integrates multiple water quality indicators to determine potability.

### 13.2) Environmental Interpretation of Key Variables

#### The feature importance results highlight pH as the most influential variable in predicting water potability. This finding is environmentally meaningful, as pH plays a fundamental role in water chemistry and treatment processes. Extreme pH levels can indicate contamination or corrosive conditions that compromise water safety.
#### Sulfate was identified as the second most important variable. Elevated sulfate concentrations may affect taste and indicate mineral or industrial contamination sources. Its relevance in the model suggests that sulfate levels contribute significantly to distinguishing potable from non-potable samples.
#### Chloramines, commonly used as disinfectants in water treatment systems, also showed strong predictive importance. Their presence reflects treatment conditions, and abnormal levels may signal inconsistencies in disinfection processes.
#### Overall, the model’s feature ranking aligns with known environmental and chemical principles of water quality assessment.

### 13.3) Model Behavior & Decision Logic

#### The optimized Gradient Boosting model makes decisions based on a combination of multiple water quality parameters rather than relying on a single dominant variable.
#### Although pH has the highest importance score, other variables such as sulfate, chloramines, hardness, and solids also contribute meaningfully to the classification process. This indicates that the model integrates various chemical indicators when determining potability.
#### Such multi-variable decision behavior enhances robustness and reduces the risk of unstable predictions caused by isolated parameter fluctuations.

### 13.4) Practical Application Scenario

In [ ]:
# Create a function

def predict_water_quality(ph, hardness, solids, chloramines, sulfate, conductivity, organic_carbon, trihalomethanes, turbidity):
    sample = pd.DataFrame({"ph": [ph], "Hardness": [hardness], "Solids": [solids], "Chloramines": [chloramines], "Sulfate": [sulfate], 
                           "Conductivity": [conductivity], "Organic_carbon": [organic_carbon], "Trihalomethanes": [trihalomethanes], 
                           "Turbidity": [turbidity]})
    prediction = best_gb.predict(sample)[0]

    if prediction == 1:
        return "The water sample is classified as POTABLE."
    else:
        return "The water sample is classified as NON-POTABLE."

In [ ]:
# Insert random values 
predict_water_quality(ph=7.2, hardness=180, solids=15000, chloramines=7.5, sulfate=320, conductivity=400, organic_carbon=12, 
                      trihalomethanes=75, turbidity=3.5)

#### To simulate real-world application, a new water sample with specific chemical parameters was introduced into the optimized Gradient Boosting model.
#### The model returned a classification output of 1, which corresponds to potable water according to the dataset encoding (0 = non-potable, 1 = potable).
#### This results demonstrates thtat the trained model can evaluate unseen water samples based on learned patterns. 
#### Although this prediction does not replace laboratory validation, it shows the model's practical capability as a preliminar screening tool for water quality assessment.ent.

### 13.5) Final Environmental Insights & Implications

#### The integration of machine learning techniques into water quality assessment demonstrates the potential of predictive modeling in environmental monitoring.
#### The optimized Gradient Boosting model effectively integrates multiple chemical indicators to classify potable water, reflecting realistic environmental interactions between parameters such as pH, sulfate, and disinfectant levels.
#### Although the model does not replace laboratory standards or regulatory frameworks, it provides a complementary analytical tool capable of supporting early risk detection and decision-making processes in water quality management.
#### This study highlights the practical value of data-driven environmental analysis in enhancing public health protection and sustainable water resource management.

## 14) Final Conclusions

### 14.1) Overall Model Performance

#### This study evaluated multiple machine learning models to classify water samples as potable or non-potable based on physicochemical parameters.
#### After hyperparameter optimization and generalization assessment:
- Random Forest achieved the highest overall accuracy.
- Support Vector Machine showed strong recall performance.
- Gradient Boosting demonstrated the best balance between F1-score, AUC, and generalization stability.
#### Considering both performance metrics and overfitting control, Gradient Boosting was selected as the final optimized model.

### 14.2) Model Reliability and Generalization

#### The comparison between training and testing accuracy showed:
- Controlled generalization gaps.
- No extreme overfitting behavior.
- Stable discrimination capacity across unseen data.
#### Although optimization did not produce dramatic performance improvements, it refined model balance and improved robustness.
#### This indicates that the dataset structure, rather than hyperparameter limitations, may constrain overall performance.

### 14.3) Environmental Interpretation

#### Feature importance analysis revealed that:
- pH
- Sulfate
- Chloramines
- Hardness
#### are among the most influential variables in water potability classification.
#### This aligns with environmental science principles, where chemical balance and contaminant levels directly affect water safety.
#### Importantly, no variable showed negligible contribution, meaning all measured parameters add predictive value.

### 14.4) Practical Applicability

#### The optimized Gradient Boosting model was successfully applied to new unseen water samples.
#### The model returned binary predictions (0 = non-potable, 1 = potable), demonstrating its practical applicability for real-world screening scenarios.
#### However, this tool should be considered a decision-support system, not a replacement for laboratory validation.

### 14.5) Final Reflection

#### This project demonstrates that:
- Machine learning can support environmental risk assessment.
- Predictive modeling enhances early detection capacity.
- Balanced evaluation (accuracy, recall, AUC, generalization) is essential for responsible model selection.
#### While performance is moderate, the structured workflow ensures transparency, interpretability, and scientific rigor.
#### Future improvements could include:
- Larger datasets
- Feature engineering
- Advanced ensemble techniques
- Cross-validation refinementalidation refinement

